In [2]:
# ============================================================================
# HW13 - BERT Fine-tuning for Emotion Classification (ФИНАЛЬНАЯ ВЕРСИЯ)
# ============================================================================

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import seaborn as sns

# 1. Настройка seed и устройства
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Загрузка данных
print("\n=== Загрузка датасета ===")
dataset_full = load_dataset("emotion", trust_remote_code=True)
dataset = {
    "train": dataset_full["train"].select(range(3000)),
    "validation": dataset_full["validation"].select(range(500)),
    "test": dataset_full["test"]
}
label_names = dataset["train"].features["label"].names
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")

# 3. Токенизация
print("\n=== Токенизация ===")
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
MAX_LENGTH = 64

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

tokenized_datasets = {
    "train": dataset["train"].map(tokenize_function, batched=True, remove_columns=["text"]),
    "validation": dataset["validation"].map(tokenize_function, batched=True, remove_columns=["text"]),
    "test": dataset["test"].map(tokenize_function, batched=True, remove_columns=["text"])
}
print(f"✓ Токенизация завершена. Length: {MAX_LENGTH}")

# 4. Инференс до обучения
print("\n=== Инференс (случайные веса) ===")
demo_model = BertForSequenceClassification.from_pretrained(model_name, num_labels=6).to(device)
demo_model.eval()
test_texts = dataset["test"]["text"][:3]
true_labels_pre = [label_names[l] for l in dataset["test"]["label"][:3]]

with torch.no_grad():
    for text, true_label in zip(test_texts, true_labels_pre):
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LENGTH).to(device)
        outputs = demo_model(**inputs)
        pred_id = torch.argmax(outputs.logits, dim=1).item()
        print(f"Text: {text[:40]}... | True: {true_label} | Pred: {label_names[pred_id]}")
print("⚠️ Предсказания случайны (модель не обучена).\n")

# 5. Fine-tuning
print("=== Настройка модели и обучение ===")
model = BertForSequenceClassification.from_pretrained(
    model_name, num_labels=6, problem_type="single_label_classification"
).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1_macro": f1_score(labels, preds, average="macro")}

training_args = TrainingArguments(
    output_dir="./results_HW13",
    eval_strategy="epoch",  # ИСПРАВЛЕНО: было evaluation_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",
    seed=42,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator
    # ИСПРАВЛЕНО: убран EarlyStoppingCallback и tokenizer
)

print("Запуск обучения...")
trainer.train()
print("✓ Обучение завершено.\n")

# 6. Финальная оценка и сохранение артефактов
print("=== Оценка на test и сохранение ===")
predictions = trainer.predict(tokenized_datasets["test"])
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

acc = accuracy_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels, average="macro")
print(f"Test Accuracy: {acc:.4f}")
print(f"Test F1 Macro: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(true_labels, pred_labels, target_names=label_names))

# Матрица ошибок
cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
plt.title("Confusion Matrix (Test Set)")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

# ИСПРАВЛЕНО: создаём папку перед сохранением
os.makedirs("homeworks/HW13/artifacts", exist_ok=True)

plt.savefig("homeworks/HW13/artifacts/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.close()

# Сохранение предсказаний
confidences = torch.softmax(torch.tensor(predictions.predictions), dim=-1).numpy()
max_confidences = np.max(confidences, axis=1)

sample_indices = random.sample(range(len(tokenized_datasets["test"])), 10)
sample_data = []
for idx in sample_indices:
    sample_data.append({
        "text": dataset["test"]["text"][idx][:200],
        "true_label": label_names[true_labels[idx]],
        "pred_label": label_names[pred_labels[idx]],
        "confidence": f"{max_confidences[idx]:.3f}"
    })

pd.DataFrame(sample_data).to_csv("homeworks/HW13/artifacts/sample_predictions.csv", index=False, encoding="utf-8")
print("\n✅ Артефакты сохранены в homeworks/HW13/artifacts/")
print("=== Готово! ===")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Using device: cpu

=== Загрузка датасета ===
Train: 3000 | Val: 500 | Test: 2000

=== Токенизация ===
✓ Токенизация завершена. Length: 64

=== Инференс (случайные веса) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Text: im feeling rather rotten so im not very ... | True: sadness | Pred: anger
Text: im updating my blog because i feel shitt... | True: sadness | Pred: anger
Text: i never make her separate from me becaus... | True: sadness | Pred: joy
⚠️ Предсказания случайны (модель не обучена).

=== Настройка модели и обучение ===


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

✓ Обучение завершено.

=== Оценка на test и сохранение ===


c:\Users\Anna\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Test Accuracy: 0.8280
Test F1 Macro: 0.6921

Classification Report:
              precision    recall  f1-score   support

     sadness       0.86      0.90      0.88       581
         joy       0.84      0.95      0.89       695
        love       0.75      0.41      0.53       159
       anger       0.78      0.81      0.80       275
        fear       0.79      0.77      0.78       224
    surprise       0.79      0.17      0.28        66

    accuracy                           0.83      2000
   macro avg       0.80      0.67      0.69      2000
weighted avg       0.82      0.83      0.81      2000


✅ Артефакты сохранены в homeworks/HW13/artifacts/
=== Готово! ===
